# Pattern 1: MLflow-native deployment to SageMaker AI

The most direct path: deploy a model **straight from the MLflow Model Registry** to a
SageMaker AI endpoint using MLflow's built-in SageMaker deployment target
([MLflow docs](https://mlflow.org/docs/latest/ml/deployment/deploy-model-to-sagemaker/)).

```
MLflow Model Registry ──mlflow.deployments──▶ SageMaker AI endpoint
```

**How it works:** MLflow packages the model with its own **pyfunc serving container**
(built and pushed to your ECR with `mlflow sagemaker build-and-push-container`), then
creates the SageMaker Model, EndpointConfig, and Endpoint for you.

**When to choose this pattern:**
- a single data science team with a single MLflow app
- MLflow is the only registry you need — approvals happen via MLflow stages/aliases
- you want the least AWS-specific code

**Trade-offs:**
- deployment does not use the **SageMaker AI Model Registry** — no IAM-gated
  lifecycle governance, no integration with SageMaker Projects/CI-CD approval events.
  (With Model Registry sync enabled — as in this repo's environment — registration
  still auto-creates a metadata-only Model Package, but this pattern never consumes it)
- you serve from MLflow's generic pyfunc container rather than the framework-optimized
  SageMaker containers
- building the container requires Docker and an ECR push — a one-time setup step

In [ ]:
%store -r mlflow_app_arn
%store -r mlflow_model_ids
%store -r model_base_name
%store -r execution_role
%store -r region
%store -r account_id

import mlflow

mlflow.set_tracking_uri(mlflow_app_arn)
client = mlflow.MlflowClient()

## Step 1: Register the model in the MLflow Model Registry

Pattern 1 works entirely inside MLflow, so we register the model logged in
`00_setup_and_train.ipynb` under a pattern-specific name.

> Because the MLflow app has auto-registration enabled, this call *also* creates a
> Model Package in the SageMaker Model Registry — we simply don't use it in this
> pattern. With auto-registration disabled, nothing outside MLflow would happen.

In [ ]:
registered_name = f"{model_base_name}-native"

mv = mlflow.register_model(f"models:/{mlflow_model_ids['native']}", registered_name)
model_uri = f"models:/{registered_name}/{mv.version}"
print(f"Registered {registered_name} v{mv.version}")
print(f"Model URI: {model_uri}")

## Step 2: Build and push the MLflow serving container (one-time)

MLflow serves the model from its own pyfunc container image. Build it once and push
it to ECR in your account:

> **Requires Docker.** In a Studio JupyterLab space, Docker must be enabled on the
> domain (`aws sagemaker update-domain ... --docker-settings EnableDockerAccess=ENABLED`)
> and the docker CLI installed. Alternatively run this cell on any Docker-capable
> machine with AWS credentials, or pass a previously pushed image via `image_url`
> below.

In [ ]:
import os

# `--network sagemaker` is required only when building inside a SageMaker
# Studio space; buildkit rejects it elsewhere (e.g. Docker Desktop on a laptop).
network_flag = "--network sagemaker" if os.path.exists("/opt/ml/metadata/resource-metadata.json") else ""

!mlflow sagemaker build-and-push-container --build --push -c mlflow-pyfunc {network_flag}

In [ ]:
mlflow_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/mlflow-pyfunc:{mlflow.__version__}"
print(f"MLflow serving image: {mlflow_image_uri}")

## Step 3: Deploy with the MLflow deployments API

`create_deployment` creates the SageMaker Model, EndpointConfig, and Endpoint in one
call and waits for the endpoint to be in service.

In [ ]:
from mlflow.deployments import get_deploy_client
from mlflow.models import get_model_info

# Check the model signature before deploying to SageMaker: without a signature
# the serving container cannot validate inference payloads.
model_info = get_model_info(model_uri)
if model_info.signature is None:
    raise ValueError(f"Model {model_uri} has no signature — refusing to deploy.")
print(f"Model signature OK: {model_info.signature}")

endpoint_name = f"{model_base_name}-native"

deploy_client = get_deploy_client(f"sagemaker:/{region}")
deploy_client.create_deployment(
    name=endpoint_name,
    model_uri=model_uri,
    config={
        "execution_role_arn": execution_role,
        "image_url": mlflow_image_uri,
        "instance_type": "ml.m5.xlarge",
        "instance_count": 1,
        "region_name": region,
        # Wait at most 10 minutes for the endpoint to become InService.
        "timeout_seconds": 600,
    },
)
print(f"Endpoint {endpoint_name} deployed.")

## Step 4: Invoke the endpoint

The pyfunc container accepts MLflow's standard inference payload formats
(`dataframe_split`, `instances`, ...).

In [ ]:
import pandas as pd

test_input = pd.DataFrame(
    [[0.5, -1.2, 0.3, 0.8], [1.1, 0.4, -0.7, 0.2]],
    columns=["f1", "f2", "f3", "f4"],
)
prediction = deploy_client.predict(endpoint_name, test_input.values)
print("Prediction:", prediction)

## What you get — and what you don't

✅ Fastest path from a registered MLflow model to a live endpoint
✅ No SageMaker-specific packaging knowledge required
✅ Model flavor handled by MLflow's pyfunc abstraction

❌ No entry in the SageMaker AI Model Registry → no cross-account sharing, no
   IAM-conditioned lifecycle stages, no approval-driven CI/CD triggers
❌ Generic pyfunc container instead of framework-optimized SageMaker containers
❌ Container build/push is on you

If your organization needs model governance beyond a single team, continue with
patterns 2 and 3, which put the **SageMaker AI Model Registry** at the center.

## Teardown (and run `04_cleanup.ipynb` at the end)

In [ ]:
# deploy_client.delete_deployment(endpoint_name)
# print(f"Deleted endpoint {endpoint_name}")